In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
INVESTIGASI & VALIDASI DATA STEAD (SUBSET 10.000 SAMPLES)
- Memeriksa struktur JSON
- Memvalidasi keberadaan P-arrival
- Menghitung statistik dasar
- Memeriksa kualitas sinyal (normalisasi, panjang, NaN)
"""

import json
import os
import numpy as np
from datetime import datetime
import matplotlib.pyplot as plt

# =============================================
# 1. KONFIGURASI
# =============================================
JSON_PATH = '/Volumes/Extreme SSD/stream_stead/data_stead/STEAD_10000_Harmonized_Final.json'
OUTPUT_DIR = '/Volumes/Extreme SSD/mcu_quake_output_replikasi_demo/investigasi_stead'

os.makedirs(OUTPUT_DIR, exist_ok=True)

# =============================================
# 2. INVESTIGASI UTAMA
# =============================================

def investigate_stead(json_path):
    print("="*80)
    print("🔍 INVESTIGASI DATASET STEAD (SUBSET 10.000)")
    print("="*80)
    
    # --- A. Cek ukuran file ---
    size_mb = os.path.getsize(json_path) / (1024**2)
    print(f"📦 Ukuran file: {size_mb:.2f} MB")
    
    # --- B. Load JSON ---
    print("📂 Memuat JSON...")
    with open(json_path, 'r') as f:
        data = json.load(f)
    
    print(f"✅ Total entri: {len(data):,}")
    
    # Cek tipe data
    if isinstance(data, list):
        print("📋 Tipe data: LIST")
        # Ambil sample pertama
        sample = data[0]
    elif isinstance(data, dict):
        print("📋 Tipe data: DICT (key-value)")
        keys = list(data.keys())
        print(f"   Contoh 5 key: {keys[:5]}")
        sample_key = keys[0]
        sample = data[sample_key]
        print(f"   Sample key: {sample_key}")
    else:
        print(f"❌ Tipe data tidak dikenal: {type(data)}")
        return
    
    # --- C. Struktur Sample ---
    print("\n" + "="*80)
    print("📋 STRUKTUR SAMPLE PERTAMA")
    print("="*80)
    print(f"Keys dalam sample: {list(sample.keys())}")
    
    # Cek apakah ada metadata
    has_metadata = 'metadata' in sample
    if has_metadata:
        meta = sample['metadata']
        print(f"Metadata keys: {list(meta.keys())}")
    else:
        print("⚠️ Tidak ada key 'metadata'. Data mungkin masih mentah (STEAD original).")
        meta = sample  # anggap metadata langsung di root
    
    # --- D. Keberadaan P-arrival ---
    print("\n" + "="*80)
    print("⏰ CEK KEBERADAAN P-ARRIVAL")
    print("="*80)
    
    has_p_arrival = False
    p_arrival_key = None
    
    # Cek di berbagai kemungkinan lokasi
    possible_keys = ['p_arrival', 'p_arrival_sample', 'trace_p_arrival', 
                     'p_time', 'P_time', 'arrival_time_P']
    
    # Cek di metadata
    if has_metadata:
        for key in possible_keys:
            if key in meta:
                has_p_arrival = True
                p_arrival_key = key
                break
    
    # Cek di root (jika tidak ada metadata)
    if not has_p_arrival:
        for key in possible_keys:
            if key in sample:
                has_p_arrival = True
                p_arrival_key = key
                break
    
    # Cek apakah ada origin_time (untuk menghitung travel time)
    has_origin = 'origin_time' in meta if has_metadata else 'origin_time' in sample
    
    if has_p_arrival:
        print(f"✅ P-arrival DITEMUKAN! Key: '{p_arrival_key}'")
        val = sample.get(p_arrival_key) if p_arrival_key in sample else meta.get(p_arrival_key)
        print(f"   Nilai sample: {val}")
    else:
        print("❌ P-arrival TIDAK DITEMUKAN.")
        print("   Mungkin data belum di-harmonisasi dengan format MCU-Quake.")
    
    # --- E. Validasi Komponen Sinyal (Z, N, E) ---
    print("\n" + "="*80)
    print("📊 VALIDASI KOMPONEN SINYAL (Z, N, E)")
    print("="*80)
    
    components = ['Z', 'N', 'E']
    noise_components = ['Z_noise', 'N_noise', 'E_noise']
    
    comp_lengths = {}
    comp_stats = {}
    
    for comp in components:
        if comp in sample:
            arr = np.array(sample[comp])
            comp_lengths[comp] = len(arr)
            comp_stats[comp] = {
                'min': np.min(arr),
                'max': np.max(arr),
                'mean': np.mean(arr),
                'std': np.std(arr),
                'has_nan': np.isnan(arr).any(),
                'has_inf': np.isinf(arr).any()
            }
            print(f"✅ {comp}: length={len(arr)}, min={np.min(arr):.4f}, max={np.max(arr):.4f}, mean={np.mean(arr):.4f}")
        else:
            print(f"❌ {comp}: TIDAK DITEMUKAN")
    
    # --- F. Validasi Noise ---
    print("\n" + "="*80)
    print("🔊 VALIDASI KOMPONEN NOISE")
    print("="*80)
    
    for noise in noise_components:
        if noise in sample:
            arr = np.array(sample[noise])
            print(f"✅ {noise}: length={len(arr)}, min={np.min(arr):.4f}, max={np.max(arr):.4f}")
        else:
            print(f"❌ {noise}: TIDAK DITEMUKAN")
    
    # --- G. Statistik Seluruh Data ---
    print("\n" + "="*80)
    print("📊 STATISTIK SELURUH DATA STEAD")
    print("="*80)
    
    total = len(data)
    have_z = 0
    have_n = 0
    have_e = 0
    have_z_noise = 0
    have_p = 0
    have_origin = 0
    z_lengths = []
    n_lengths = []
    e_lengths = []
    z_noise_lengths = []
    
    # Untuk sampling, kita loop semua data
    print("⏳ Memproses semua entri...")
    for i, (key, record) in enumerate(data.items() if isinstance(data, dict) else enumerate(data)):
        if i % 1000 == 0 and i > 0:
            print(f"   Diproses {i:,} entri...")
        
        if 'Z' in record:
            have_z += 1
            z_lengths.append(len(record['Z']))
        if 'N' in record:
            have_n += 1
            n_lengths.append(len(record['N']))
        if 'E' in record:
            have_e += 1
            e_lengths.append(len(record['E']))
        if 'Z_noise' in record:
            have_z_noise += 1
            z_noise_lengths.append(len(record['Z_noise']))
        
        # Cek P-arrival (di metadata atau root)
        meta = record.get('metadata', {})
        if 'p_arrival' in meta or 'p_arrival' in record:
            have_p += 1
        if 'origin_time' in meta or 'origin_time' in record:
            have_origin += 1
    
    print(f"\n📊 Statistik Komponen Sinyal:")
    print(f"   Z: {have_z}/{total} ({have_z/total*100:.2f}%)")
    print(f"   N: {have_n}/{total} ({have_n/total*100:.2f}%)")
    print(f"   E: {have_e}/{total} ({have_e/total*100:.2f}%)")
    print(f"   Z_noise: {have_z_noise}/{total} ({have_z_noise/total*100:.2f}%)")
    
    if z_lengths:
        uniq_z = set(z_lengths)
        print(f"   Panjang Z: {sorted(uniq_z)} (unique)")
    if n_lengths:
        uniq_n = set(n_lengths)
        print(f"   Panjang N: {sorted(uniq_n)} (unique)")
    if e_lengths:
        uniq_e = set(e_lengths)
        print(f"   Panjang E: {sorted(uniq_e)} (unique)")
    
    print(f"\n⏰ P-arrival tersedia: {have_p}/{total} ({have_p/total*100:.2f}%)")
    print(f"⏳ Origin time tersedia: {have_origin}/{total} ({have_origin/total*100:.2f}%)")
    
    # --- H. Kesimpulan ---
    print("\n" + "="*80)
    print("💡 KESIMPULAN INVESTIGASI")
    print("="*80)
    
    if have_p > 0 and have_origin > 0:
        print("✅ Data STEAD ini MENGANDUNG P-arrival dan Origin Time.")
        print("   → Data SIAP untuk validasi travel time dan benchmarking.")
    elif have_p > 0:
        print("⚠️ Data memiliki P-arrival tetapi TIDAK ADA origin_time.")
        print("   → Travel time tidak bisa dihitung.")
    else:
        print("⚠️ Data TIDAK memiliki P-arrival atau origin_time.")
        print("   → Ini mungkin data mentah STEAD (belum di-harmonisasi).")
    
    if have_z == total and have_n == total and have_e == total:
        print("✅ Data memiliki 3 komponen (Z, N, E) lengkap untuk semua entri.")
    else:
        print("⚠️ Beberapa entri tidak memiliki 3 komponen lengkap.")
    
    if z_lengths and all(l == 700 for l in z_lengths):
        print("✅ Panjang Z = 700 (sesuai MCU-Quake).")
    elif z_lengths and all(l == 6000 for l in z_lengths):
        print("⚠️ Panjang Z = 6000 (STEAD original, belum dipotong ke 7 detik).")
    else:
        print(f"⚠️ Panjang Z bervariasi: {set(z_lengths)}")
    
    # --- I. Simpan Statistik ke CSV ---
    import pandas as pd
    stats_df = pd.DataFrame({
        'total_entries': [total],
        'have_Z': [have_z],
        'have_N': [have_n],
        'have_E': [have_e],
        'have_Z_noise': [have_z_noise],
        'have_p_arrival': [have_p],
        'have_origin_time': [have_origin],
        'Z_length': [list(set(z_lengths)) if z_lengths else 'N/A'],
    })
    csv_path = os.path.join(OUTPUT_DIR, 'stead_investigation_stats.csv')
    stats_df.to_csv(csv_path, index=False)
    print(f"\n✅ Statistik tersimpan: {csv_path}")
    
    return data, sample

# =============================================
# 3. EKSEKUSI
# =============================================

if __name__ == "__main__":
    data, sample = investigate_stead(JSON_PATH)

🔍 INVESTIGASI DATASET STEAD (SUBSET 10.000)
📦 Ukuran file: 7421.07 MB
📂 Memuat JSON...
✅ Total entri: 20,000
📋 Tipe data: DICT (key-value)
   Contoh 5 key: ['ASBU.CC_201309080500_NO', 'AMKA.AV_20140727033855_EV', 'NAGD.HV_20180507175116_EV', 'CRY.AZ_20040319010929_EV', 'AP01.C1_20180115025048_NO']
   Sample key: ASBU.CC_201309080500_NO

📋 STRUKTUR SAMPLE PERTAMA
Keys dalam sample: ['type', 'Z', 'N', 'E', 'Z_noise', 'N_noise', 'E_noise', 'norm']
⚠️ Tidak ada key 'metadata'. Data mungkin masih mentah (STEAD original).

⏰ CEK KEBERADAAN P-ARRIVAL
❌ P-arrival TIDAK DITEMUKAN.
   Mungkin data belum di-harmonisasi dengan format MCU-Quake.

📊 VALIDASI KOMPONEN SINYAL (Z, N, E)
✅ Z: length=5000, min=-0.6713, max=0.6180, mean=-0.0000
✅ N: length=5000, min=-0.6713, max=0.6180, mean=-0.0000
✅ E: length=5000, min=-0.6713, max=0.6180, mean=-0.0000

🔊 VALIDASI KOMPONEN NOISE
✅ Z_noise: length=1000, min=-0.5102, max=0.4657
✅ N_noise: length=1000, min=-0.5102, max=0.4657
✅ E_noise: length=1000, min=-0

In [2]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
INVESTIGASI ORIGINAL STEAD (merge.csv & merge.hdf5)
- Membaca metadata (CSV) untuk melihat kolom dan label P-arrival
- Membaca struktur HDF5 untuk melihat bentuk data
- Memeriksa ketersediaan p_arrival_sample
"""

import os
import sys
import pandas as pd
import h5py
import numpy as np

# =============================================
# 1. KONFIGURASI
# =============================================
CSV_PATH = '/Volumes/Extreme SSD/stream_stead/data_stead/merge.csv'
HDF5_PATH = '/Volumes/Extreme SSD/stream_stead/data_stead/merge.hdf5'

# =============================================
# 2. CEK UKURAN FILE
# =============================================
print("="*80)
print("📂 INVESTIGASI ORIGINAL STEAD")
print("="*80)

if os.path.exists(CSV_PATH):
    size_gb = os.path.getsize(CSV_PATH) / (1024**3)
    print(f"✅ CSV ditemukan: {size_gb:.2f} GB")
else:
    print(f"❌ CSV tidak ditemukan: {CSV_PATH}")
    sys.exit(1)

if os.path.exists(HDF5_PATH):
    size_gb = os.path.getsize(HDF5_PATH) / (1024**3)
    print(f"✅ HDF5 ditemukan: {size_gb:.2f} GB")
else:
    print(f"❌ HDF5 tidak ditemukan: {HDF5_PATH}")
    sys.exit(1)

# =============================================
# 3. INSPEKSI CSV (METADATA)
# =============================================
print("\n" + "="*80)
print("📋 INSPEKSI METADATA (CSV)")
print("="*80)

# Baca hanya 5 baris pertama untuk melihat struktur (hindari loading besar)
try:
    df_sample = pd.read_csv(CSV_PATH, nrows=5)
    print(f"✅ Jumlah kolom: {len(df_sample.columns)}")
    print(f"📋 Nama kolom (sample): {df_sample.columns.tolist()[:20]} ... (dan seterusnya)")
    
    # Cek keberadaan P-arrival
    has_p = 'p_arrival_sample' in df_sample.columns
    has_s = 's_arrival_sample' in df_sample.columns
    has_origin = 'source_origin_time' in df_sample.columns
    has_mag = 'source_magnitude' in df_sample.columns
    
    print("\n🔍 Kolom penting:")
    print(f"   p_arrival_sample: {'✅ ADA' if has_p else '❌ TIDAK ADA'}")
    print(f"   s_arrival_sample: {'✅ ADA' if has_s else '❌ TIDAK ADA'}")
    print(f"   source_origin_time: {'✅ ADA' if has_origin else '❌ TIDAK ADA'}")
    print(f"   source_magnitude: {'✅ ADA' if has_mag else '❌ TIDAK ADA'}")
    
    if has_p:
        print(f"   Contoh nilai p_arrival_sample: {df_sample['p_arrival_sample'].tolist()}")
    
    # Cek trace_category
    if 'trace_category' in df_sample.columns:
        categories = df_sample['trace_category'].value_counts()
        print(f"\n📊 Distribusi trace_category (5 sample): {df_sample['trace_category'].tolist()}")
    
    # Cek total baris (estimasi)
    try:
        # Baca hanya kolom pertama untuk hitung baris (lebih cepat)
        total_rows = pd.read_csv(CSV_PATH, usecols=[0]).shape[0]
        print(f"\n📊 Estimasi total entri: {total_rows:,}")
    except:
        print("\n⚠️ Tidak bisa menghitung total baris (file terlalu besar?)")
        
except Exception as e:
    print(f"❌ Gagal membaca CSV: {e}")

# =============================================
# 4. INSPEKSI HDF5 (WAVEFORM)
# =============================================
print("\n" + "="*80)
print("📊 INSPEKSI WAVEFORM (HDF5)")
print("="*80)

try:
    with h5py.File(HDF5_PATH, 'r') as hf:
        # Tampilkan semua keys di root
        print(f"🔑 Keys dalam HDF5: {list(hf.keys())}")
        
        # Biasanya data disimpan di key 'data' atau 'waveforms'
        if 'data' in hf:
            dataset = hf['data']
            print(f"✅ Dataset 'data' ditemukan!")
            print(f"   Shape: {dataset.shape}")
            print(f"   Dtype: {dataset.dtype}")
            
            # Shape biasanya (n_samples, 3, 6000) atau (n_samples, 6000, 3)
            if len(dataset.shape) == 3:
                n_samples, dim1, dim2 = dataset.shape
                print(f"   Jumlah sampel: {n_samples}")
                print(f"   Dimensi 1: {dim1}")
                print(f"   Dimensi 2: {dim2}")
                
                # Tentukan mana komponen dan mana waktu
                if dim1 == 3 and dim2 == 6000:
                    print("   ✅ Format: (n_samples, 3 komponen, 6000 sampel) -> STEAD original!")
                elif dim1 == 6000 and dim2 == 3:
                    print("   ✅ Format: (n_samples, 6000 sampel, 3 komponen)")
                else:
                    print(f"   ⚠️ Format tidak standar: {dataset.shape}")
        
        elif 'waveforms' in hf:
            dataset = hf['waveforms']
            print(f"✅ Dataset 'waveforms' ditemukan!")
            print(f"   Shape: {dataset.shape}")
        
        else:
            # Coba cari dataset apa saja
            print("🔍 Mencari dataset di semua level...")
            def print_dataset(name, obj):
                if isinstance(obj, h5py.Dataset):
                    print(f"   - {name}: {obj.shape} {obj.dtype}")
            hf.visititems(print_dataset)
            
except Exception as e:
    print(f"❌ Gagal membaca HDF5: {e}")

# =============================================
# 5. KESIMPULAN
# =============================================
print("\n" + "="*80)
print("💡 KESIMPULAN")
print("="*80)

if has_p and has_origin:
    print("🎉 STEAD ORIGINAL INI SANGAT LENGKAP!")
    print("   ✅ Ada p_arrival_sample (label P-wave).")
    print("   ✅ Ada origin_time.")
    print("   ✅ Ada waveform 3C 60 detik.")
    print("\n📌 Data ini SIAP digunakan untuk:")
    print("   1. Ekstraksi 7 detik sinyal setelah P.")
    print("   2. Ekstraksi 7 detik noise sebelum P.")
    print("   3. Benchmarking MCU-Quake dengan label manual/AI (akurat!).")
    print("   4. Validasi hasil STA/LTA Anda sebelumnya.")
else:
    print("⚠️ Data ini mungkin belum lengkap atau formatnya berbeda.")
    print("   Pastikan Anda mendownload STEAD versi lengkap dari GitHub resmi.")

📂 INVESTIGASI ORIGINAL STEAD
✅ CSV ditemukan: 0.33 GB
✅ HDF5 ditemukan: 90.99 GB

📋 INSPEKSI METADATA (CSV)
✅ Jumlah kolom: 35
📋 Nama kolom (sample): ['network_code', 'receiver_code', 'receiver_type', 'receiver_latitude', 'receiver_longitude', 'receiver_elevation_m', 'p_arrival_sample', 'p_status', 'p_weight', 'p_travel_sec', 's_arrival_sample', 's_status', 's_weight', 'source_id', 'source_origin_time', 'source_origin_uncertainty_sec', 'source_latitude', 'source_longitude', 'source_error_sec', 'source_gap_deg'] ... (dan seterusnya)

🔍 Kolom penting:
   p_arrival_sample: ✅ ADA
   s_arrival_sample: ✅ ADA
   source_origin_time: ✅ ADA
   source_magnitude: ✅ ADA
   Contoh nilai p_arrival_sample: [nan, nan, nan, nan, nan]

📊 Distribusi trace_category (5 sample): ['noise', 'noise', 'noise', 'noise', 'noise']

📊 Estimasi total entri: 1,265,657

📊 INSPEKSI WAVEFORM (HDF5)
🔑 Keys dalam HDF5: ['data']
✅ Dataset 'data' ditemukan!
❌ Gagal membaca HDF5: 'Group' object has no attribute 'shape'

💡 KES